# Track 01 Workflow Notebook

This notebook follows the [analysis workflow](../3-workflow.md) step by step. Each numbered section matches a step in that guide, so read them side by side.

The notebook handles the coding. Your team makes the decisions. Cells you need to edit are marked **You decide**. Everything else runs as is.

Run cells from top to bottom with Shift+Enter.

## 0. Setup

Works on your own computer and on Google Colab. On Colab this cell downloads the repository and installs PyDESeq2, which takes about a minute.

In [ ]:
import os, sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/openbio-datathon-2026").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Harvard-Undergraduate-OpenBio/openbio-datathon-2026.git", "/content/openbio-datathon-2026"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydeseq2"], check=True)
    os.chdir("/content/openbio-datathon-2026/track-01/notebooks")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

DATA_DIR = Path("../data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
print("Setup complete. Data folder:", DATA_DIR.resolve())

## 1. Pick a Question

Choose one from the [starter questions](../starter-questions.md) and write it here. Double-click this cell to edit it.

**You decide:** Our question is ...

## 2. Load and Check the Data

Loads the count table (genes by samples) and the metadata table (one row per sample), then checks that they fit together. If a check prints `False` or a nonzero duplicate count, note it in your analysis plan and tell an organizer. Do not edit the data files.

In [ ]:
def load_table(path):
    if not path.exists() or path.stat().st_size == 0:
        print(f"{path.name}: pending release (file is empty for now).")
        return None
    return pd.read_csv(path, index_col=0)

counts = load_table(DATA_DIR / "track01_counts_vt.csv")
metadata = load_table(DATA_DIR / "track01_metadata_vt.csv")

if counts is not None and metadata is not None:
    print(f"Counts: {counts.shape[0]:,} genes x {counts.shape[1]} samples")
    print(f"Metadata: {metadata.shape[0]} samples x {metadata.shape[1]} fields")
    print()
    print("Samples in both tables:      ", len(set(counts.columns) & set(metadata.index)))
    print("Counts are nonnegative:      ", bool((counts.values >= 0).all()))
    print("Counts are whole numbers:    ", bool((counts.values % 1 == 0).all()))
    print("Duplicate genes:             ", int(counts.index.duplicated().sum()))
    print("Duplicate samples:           ", int(counts.columns.duplicated().sum()))
    # Put metadata rows in the same order as the count columns
    metadata = metadata.loc[counts.columns]

Look at the metadata fields and how many values are missing in each. The [data dictionary](../data/track01_data_dictionary.md) explains every column.

In [ ]:
if metadata is not None:
    overview = pd.DataFrame({
        "example_value": metadata.iloc[0],
        "n_missing": metadata.isna().sum(),
        "n_unique": metadata.nunique(),
    })
    display(overview)

## 3. Explore the Samples

The PCA plot places samples with similar overall gene activity close together. Change `COLOR_BY` to different metadata columns and rerun the cell to see which fields line up with the main patterns.

Write what you notice in your analysis plan. This guides Step 4.

In [ ]:
# You decide: a metadata column name to color the points by
COLOR_BY = None

def plot_pca(counts, metadata, color_by=None):
    # Scale each sample to counts per million, then log transform so a few huge genes do not dominate
    log_cpm = np.log2(counts.div(counts.sum(axis=0), axis=1) * 1e6 + 1)
    pca = PCA(n_components=2)
    scores = pca.fit_transform(log_cpm.T)
    fig, ax = plt.subplots(figsize=(7, 5))
    if color_by is None:
        ax.scatter(scores[:, 0], scores[:, 1])
    else:
        values = metadata[color_by]
        if pd.api.types.is_numeric_dtype(values):
            points = ax.scatter(scores[:, 0], scores[:, 1], c=values, cmap="viridis")
            fig.colorbar(points, ax=ax, label=color_by)
        else:
            for level in values.fillna("NA").unique():
                mask = (values.fillna("NA") == level).values
                ax.scatter(scores[mask, 0], scores[mask, 1], label=str(level))
            ax.legend(title=color_by, bbox_to_anchor=(1.02, 1), loc="upper left")
    for i, sample in enumerate(counts.columns):
        ax.annotate(sample, scores[i], fontsize=7, alpha=0.6)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variation)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} of variation)")
    ax.set_title("Samples by overall gene activity")
    plt.tight_layout()
    plt.show()

if counts is not None:
    plot_pca(counts, metadata, COLOR_BY)

## 4. Choose a Comparison

Define the two groups you will compare. Use exact column names and values from the metadata table (Section 2 lists them).

- `GROUP_COLUMN`: the column that defines your groups
- `CASE` and `CONTROL`: the two values of that column to compare
- `COVARIATES`: other columns to account for, such as gestational age. Leave as `[]` for none.

Write a short justification for these choices in your analysis plan.

In [ ]:
# You decide
GROUP_COLUMN = None   # e.g. "column_name"
CASE = None           # e.g. "value_for_group_A"
CONTROL = None        # e.g. "value_for_group_B"
COVARIATES = []       # e.g. ["another_column"]

def build_comparison(counts, metadata):
    if None in (GROUP_COLUMN, CASE, CONTROL):
        print("Fill in GROUP_COLUMN, CASE and CONTROL above, then rerun this cell.")
        return None, None
    used = [GROUP_COLUMN] + list(COVARIATES)
    meta = metadata[metadata[GROUP_COLUMN].isin([CASE, CONTROL])][used].copy()
    n_before = len(meta)
    meta = meta.dropna()
    if len(meta) < n_before:
        print(f"Note: {n_before - len(meta)} sample(s) dropped for missing values in {used}. Report this.")
    meta[GROUP_COLUMN] = meta[GROUP_COLUMN].astype(str)
    print(meta[GROUP_COLUMN].value_counts().to_string())
    return counts[meta.index], meta

comp_counts, comp_meta = (None, None) if counts is None else build_comparison(counts, metadata)

## 5. Run the Test

Runs differential expression with PyDESeq2. It adjusts for sequencing depth, then tests every gene for a difference between `CASE` and `CONTROL` that is larger than the natural variation between samples. Expect a few seconds to a few minutes.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

results = None
if comp_counts is not None:
    design = "~" + " + ".join(list(COVARIATES) + [GROUP_COLUMN])
    print("Design:", design)
    dds = DeseqDataSet(
        counts=comp_counts.T.astype(int),   # PyDESeq2 wants samples as rows
        metadata=comp_meta,
        design=design,
        quiet=True,
    )
    dds.deseq2()
    stats = DeseqStats(dds, contrast=[GROUP_COLUMN, str(CASE), str(CONTROL)], quiet=True)
    stats.summary()
    results = stats.results_df.sort_values("padj")
    results.to_csv(RESULTS_DIR / "de_results.csv")
    print(f"Tested {len(results):,} genes. Saved to results/de_results.csv")
    display(results.head(10))

## 6. Read the Results

- `log2FoldChange`: +1 means twice as high in CASE, −1 means half as high
- `padj`: adjusted p-value, smaller means stronger evidence
- `baseMean`: average count, very low values are unreliable

Set your own cutoffs. Try a few values and note how much the number of passing genes changes.

In [ ]:
# You decide
PADJ_CUTOFF = None   # e.g. a value between 0 and 1
LFC_CUTOFF = None    # e.g. a minimum absolute log2 fold change, or 0

def passing_genes(results):
    if PADJ_CUTOFF is None or LFC_CUTOFF is None:
        return None
    keep = (results["padj"] < PADJ_CUTOFF) & (results["log2FoldChange"].abs() >= LFC_CUTOFF)
    return results[keep]

def volcano(results, hits):
    fig, ax = plt.subplots(figsize=(7, 5))
    y = -np.log10(results["padj"].clip(lower=1e-300))
    ax.scatter(results["log2FoldChange"], y, s=4, color="lightgray")
    if hits is not None:
        ax.scatter(hits["log2FoldChange"], -np.log10(hits["padj"].clip(lower=1e-300)), s=6, color="crimson", label=f"{len(hits)} passing genes")
        ax.axhline(-np.log10(PADJ_CUTOFF), ls="--", lw=0.8, color="black")
        ax.legend()
    ax.set_xlabel(f"log2 fold change ({CASE} vs {CONTROL})")
    ax.set_ylabel("-log10 adjusted p-value")
    ax.set_title("Volcano plot")
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / "volcano.png", dpi=300)
    plt.show()

hits = None
if results is not None:
    hits = passing_genes(results)
    if hits is None:
        print("Set PADJ_CUTOFF and LFC_CUTOFF to highlight genes.")
    else:
        print(f"{len(hits)} genes pass ({(hits['log2FoldChange'] > 0).sum()} higher in {CASE}, {(hits['log2FoldChange'] < 0).sum()} lower).")
        hits.to_csv(RESULTS_DIR / "passing_genes.csv")
    volcano(results, hits)

## 7. Connect Genes to Cell Types

First, check which passing genes are known markers of broad placental cell types. If many markers of one cell type move in the same direction, a change in cell mix is one possible explanation.

Gene identifiers in the counts table may not be gene symbols. If no genes match, check the data dictionary for how to map them.

In [ ]:
markers = pd.read_csv(DATA_DIR / "interpretation" / "track01_marker_matrix.csv")

if hits is not None:
    overlap = markers.merge(hits[["log2FoldChange", "padj"]], left_on="gene_symbol", right_index=True)
    if overlap.empty:
        print("None of your passing genes are in the marker table.")
    else:
        display(overlap.sort_values(["cell_class", "log2FoldChange"]))
else:
    print("Marker table loaded:", markers["cell_class"].nunique(), "cell classes. Set cutoffs in Section 6 to compare.")

Next, look up at least three of your genes in the [Zeisel/Admati gene browser](https://storage.googleapis.com/www_zeisellab/PE_web/pe_web_test_v1.html) (required). Record what you see in the table below; it is saved to `results/browser_lookups.csv` for your submission.

In [ ]:
# You decide: one row per gene you looked up
browser_lookups = pd.DataFrame([
    # {"gene": "GENE_A", "cell_types_expressing": "...", "differs_PE_vs_control": "...", "notes": "..."},
])
if not browser_lookups.empty:
    browser_lookups.to_csv(RESULTS_DIR / "browser_lookups.csv", index=False)
    display(browser_lookups)
else:
    print("Add at least three genes to browser_lookups.")

## 8. Optional: Single-Cell Extension

The pseudobulk files hold counts summed by donor and cell type. To test one cell type, filter both tables to that cell type and rerun Sections 4 to 6 with `counts` and `metadata` replaced by the filtered tables. The data dictionary describes the column layout. Each donor counts once, and the early-onset comparison is exploratory.

In [ ]:
pb_counts = load_table(DATA_DIR / "extension" / "pseudobulk" / "track01_pseudobulk_counts.csv")
pb_metadata = load_table(DATA_DIR / "extension" / "pseudobulk" / "track01_pseudobulk_metadata.csv")
if pb_counts is not None:
    print("Pseudobulk counts:", pb_counts.shape)
    display(pb_metadata.head())

## Next

Go to [Step 4: Submit](../4-submit.md). Save the figures you want to use, and fill in the hypothesis card.